# Blob vs Filament TDA

Topological Data Analysis to distinguish **compact blobs** from **filamentary/spiky manifolds** in high-dimensional point clouds.

**Primary diagnostics** (distance-based geometry):
1. **Vietoris–Rips H₀/H₁ + MST merge distribution:** blobs connect quickly at small $\varepsilon$ (few long H₀ bars, MST edges concentrated); filaments connect slowly (many long H₀ bars, heavy MST tail). H₁ loops appear when filaments create cycles.
2. **Intrinsic dimension estimate:** blob fills $\sim D$ dimensions; filament is $\sim$1-D / branched.
3. **Mapper graph** (PC1 + kNN-radius): blob → compact, few-node graph; filament → long chain with branches.

**Secondary diagnostic** (density / cluster-tree):
4. **kNN-radius sublevel H₀:** useful for detecting multiple density modes, but less clean than Rips for measuring geometric filamentness.

5. Summary dashboard

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import KDTree
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from ripser import ripser
from persim import plot_diagrams
import kmapper as km
import networkx as nx
import warnings

warnings.filterwarnings("ignore")
np.random.seed(42)

K_NN = 15  # default kNN parameter (used throughout)
PCA_DIM = 15

def knn_radius(X, k=K_NN):
    """Distance from each point to its k-th nearest neighbour."""
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    dists, _ = nn.kneighbors(X)
    return dists[:, -1]

def intrinsic_dim_knn(X, k1=5, k2=20):
    """Intrinsic dimension via kNN distance ratios (FSA estimator)."""
    nn = NearestNeighbors(n_neighbors=k2 + 1).fit(X)
    dists, _ = nn.kneighbors(X)
    r1, r2 = dists[:, k1], dists[:, k2]
    mask = (r1 > 0) & (r2 > 0)
    d_hat = np.log(k2 / k1) / np.log(r2[mask] / r1[mask])
    return np.median(d_hat)

def knn_radius_tail_ratio(X, k=K_NN, lo=50, hi=99):
    rho = knn_radius(X, k)
    return np.percentile(rho, hi) / np.percentile(rho, lo)

def rips_summary(rips_result, dim):
    dgm = rips_result["dgms"][dim]
    fin = dgm[np.isfinite(dgm[:, 1])]
    pers = (fin[:, 1] - fin[:, 0]) if len(fin) else np.array([0.0])
    return len(fin), pers.mean(), pers.max(), np.percentile(pers, 90)

def mst_edges(X_sub):
    """MST edge weights from pairwise distances (= Rips H0 death times)."""
    D = squareform(pdist(X_sub))
    tree = minimum_spanning_tree(D)
    return np.sort(tree.data)

print("All imports + utilities OK")

## 1. Point Cloud Generation

Two datasets in $[0,1]^D$:
- **Blob**: isotropic Gaussian centred at $0.5$, clipped to the unit cube
- **Filament**: union of random line segments ("spikes") with Gaussian thickening — produces a spiky, branching manifold

In [ ]:
D = 20          # ambient dimension
N = 800         # points per dataset

# --- Blob: isotropic Gaussian clipped to [0,1]^D ---
blob_raw = 0.5 + 0.12 * np.random.randn(N, D)
blob = np.clip(blob_raw, 0, 1)

# --- Filament / spiky manifold ---
# Build a random tree of segments from a hub near the centre,
# then sample points along them with thin Gaussian noise.
def make_filament(n_points, dim, n_branches=12, noise_std=0.02, rng=None):
    rng = rng or np.random.default_rng(0)
    hub = rng.uniform(0.3, 0.7, size=dim)
    pts = []
    remaining = n_points
    for i in range(n_branches):
        n_seg = remaining // (n_branches - i)
        remaining -= n_seg
        tip = rng.uniform(0.0, 1.0, size=dim)
        t = rng.uniform(0, 1, size=(n_seg, 1))
        seg = hub[None, :] * (1 - t) + tip[None, :] * t
        seg += noise_std * rng.standard_normal(seg.shape)
        pts.append(seg)
    pts = np.vstack(pts)
    np.clip(pts, 0, 1, out=pts)
    return pts

filament = make_filament(N, D, n_branches=12, noise_std=0.02,
                         rng=np.random.default_rng(7))

# --- Quick 2-D PCA preview ---
pca2 = PCA(n_components=2)
blob_2d = pca2.fit_transform(blob)
fil_2d  = PCA(n_components=2).fit_transform(filament)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
axes[0].scatter(*blob_2d.T, s=4, alpha=0.5)
axes[0].set_title("Blob (PCA 2-D)")
axes[1].scatter(*fil_2d.T, s=4, alpha=0.5, color="C1")
axes[1].set_title("Filament (PCA 2-D)")
for ax in axes:
    ax.set_aspect("equal"); ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

print(f"Blob shape: {blob.shape}, Filament shape: {filament.shape}")

## 2. Vietoris–Rips PH + MST Merge Distribution (Primary Geometric Diagnostic)

Standard distance-based persistent homology on a PCA-reduced embedding (15 dims).

- **Rips H₀(ε):** as the distance threshold $\varepsilon$ grows, connected components merge. The distribution of merge distances (= H₀ death times = MST edge weights) directly measures how quickly the point cloud connects.  
  - **Blob** → concentrated merge distances, one dominant component forms quickly.  
  - **Filament** → spread-out merge distances with a heavy tail (long gaps between branches).
- **Rips H₁:** 1-cycles (loops). Filamentary/branching data can produce longer-lived loops than a convex blob.
- **MST edge-weight distribution:** equivalent information to H₀, plotted as a histogram for clarity.

In [ ]:
blob_pca = PCA(n_components=PCA_DIM).fit_transform(blob)
fil_pca  = PCA(n_components=PCA_DIM).fit_transform(filament)

SUB = 400
idx_b = np.random.choice(len(blob_pca), min(SUB, len(blob_pca)), replace=False)
idx_f = np.random.choice(len(fil_pca),  min(SUB, len(fil_pca)),  replace=False)

rips_blob = ripser(blob_pca[idx_b], maxdim=1)
rips_fil  = ripser(fil_pca[idx_f],  maxdim=1)

mst_blob = mst_edges(blob_pca[idx_b])
mst_fil  = mst_edges(fil_pca[idx_f])

# --- Plots ---
fig = plt.figure(figsize=(16, 9))
gs = GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

ax_pd_b = fig.add_subplot(gs[0, 0])
plot_diagrams(rips_blob["dgms"], ax=ax_pd_b, show=False)
ax_pd_b.set_title("Blob – Rips persistence diagram")

ax_pd_f = fig.add_subplot(gs[0, 1])
plot_diagrams(rips_fil["dgms"], ax=ax_pd_f, show=False)
ax_pd_f.set_title("Filament – Rips persistence diagram")

ax_bars = fig.add_subplot(gs[0, 2])
for rips, lbl, col, offset in [(rips_blob, "Blob", "C0", 0),
                                 (rips_fil, "Filament", "C1", 0.4)]:
    dgm0 = rips["dgms"][0]
    fin = dgm0[np.isfinite(dgm0[:, 1])]
    pers = np.sort(fin[:, 1] - fin[:, 0])[::-1][:50]
    y = np.arange(len(pers)) + offset
    ax_bars.barh(y, pers, height=0.35, color=col, alpha=0.7, label=lbl)
ax_bars.invert_yaxis()
ax_bars.set_xlabel("Persistence"); ax_bars.set_ylabel("Bar rank")
ax_bars.set_title("Rips H₀ top-50 bars"); ax_bars.legend(fontsize=9)

ax_mst = fig.add_subplot(gs[1, 0])
combined_mst = np.concatenate([mst_blob, mst_fil])
bins_mst = np.linspace(0, np.percentile(combined_mst, 99), 50)
ax_mst.hist(mst_blob, bins=bins_mst, alpha=0.6, label="Blob", density=True)
ax_mst.hist(mst_fil,  bins=bins_mst, alpha=0.6, label="Filament", density=True)
ax_mst.set_xlabel("MST edge weight (= Rips H₀ death)")
ax_mst.set_ylabel("Density"); ax_mst.set_title("MST merge-distance distribution")
ax_mst.legend(fontsize=9)

ax_mst_log = fig.add_subplot(gs[1, 1])
lo = min(mst_blob.min(), mst_fil.min()) * 0.8
hi = max(mst_blob.max(), mst_fil.max()) * 1.2
bins_log = np.geomspace(max(lo, 1e-6), hi, 40)
ax_mst_log.hist(mst_blob, bins=bins_log, alpha=0.6, label="Blob")
ax_mst_log.hist(mst_fil,  bins=bins_log, alpha=0.6, label="Filament")
ax_mst_log.set_xscale("log"); ax_mst_log.set_yscale("log")
ax_mst_log.set_xlabel("MST edge weight (log)"); ax_mst_log.set_ylabel("Count (log)")
ax_mst_log.set_title("MST merge-distance (log-log)"); ax_mst_log.legend(fontsize=9)

ax_h1 = fig.add_subplot(gs[1, 2])
for rips, lbl, col in [(rips_blob, "Blob", "C0"), (rips_fil, "Filament", "C1")]:
    dgm1 = rips["dgms"][1]
    fin1 = dgm1[np.isfinite(dgm1[:, 1])]
    pers1 = np.sort(fin1[:, 1] - fin1[:, 0])[::-1][:30]
    ax_h1.barh(np.arange(len(pers1)), pers1, height=0.8, color=col, alpha=0.6, label=lbl)
ax_h1.invert_yaxis()
ax_h1.set_xlabel("Persistence"); ax_h1.set_ylabel("Bar rank")
ax_h1.set_title("Rips H₁ top-30 bars"); ax_h1.legend(fontsize=9)

plt.show()

for label, rips in [("Blob", rips_blob), ("Filament", rips_fil)]:
    for dim in [0, 1]:
        n_bars, mu, mx, p90 = rips_summary(rips, dim)
        print(f"  {label} H{dim}: {n_bars:4d} finite bars, "
              f"mean pers = {mu:.4f}, max pers = {mx:.4f}, p90 = {p90:.4f}")

print(f"\nMST edge-weight statistics:")
for lbl, mst in [("Blob", mst_blob), ("Filament", mst_fil)]:
    print(f"  {lbl}: median = {np.median(mst):.4f}, "
          f"mean = {mst.mean():.4f}, max = {mst.max():.4f}, "
          f"p90/p50 = {np.percentile(mst, 90)/np.median(mst):.3f}")

## 3. Intrinsic Dimension Estimate

Non-topological but strongly geometric:
- **Intrinsic dimension** (Farahmand–Szepesvári–Audibert via kNN distance ratios): blob occupies $\sim D$ dimensions; filament is $\sim$1-D.
- **kNN-radius tail ratio** (p99/p50): blob has uniform-ish density (ratio ≈ 1); filament has sparse tails (larger ratio).

In [ ]:
print(f"{'':20s} {'Intrinsic dim':>14s} {'kNN-radius p99/p50':>20s}")
print("-" * 56)
for lbl, X in [("Blob", blob), ("Filament", filament)]:
    d_est = intrinsic_dim_knn(X)
    ratio = knn_radius_tail_ratio(X)
    print(f"  {lbl:16s} {d_est:14.2f} {ratio:20.3f}")

## 4. Mapper Graph (Geometric Structure)

Two filters per dataset:
- **Filter 1 (structural):** PC1 projection — captures the dominant axis of elongation
- **Filter 2 (density):** kNN-radius — separates dense cores from sparse tails

Covering: overlapping intervals; clustering: DBSCAN in each pull-back.

**Expected:** Blob → compact, few-node graph. Filament → elongated chain with branches (spikes radiate from the hub).

In [ ]:
def run_mapper(X, label, k=K_NN, n_cubes=10, overlap=0.4, ax=None):
    """Build a Mapper graph with PC1 + kNN-radius as dual lens."""
    pc1 = PCA(n_components=1).fit_transform(X).ravel()
    rho = knn_radius(X, k)
    lens = np.column_stack([pc1, rho])

    # Adaptive DBSCAN eps: median kNN distance in full space
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    dists, _ = nn.kneighbors(X)
    eps_adapt = float(np.median(dists[:, -1]) * 1.5)

    mapper = km.KeplerMapper(verbose=0)
    graph = mapper.map(
        lens, X,
        cover=km.Cover(n_cubes=n_cubes, perc_overlap=overlap),
        clusterer=DBSCAN(eps=eps_adapt, min_samples=3),
    )

    G = nx.Graph()
    node_sizes = {}
    for node_id, members in graph["nodes"].items():
        G.add_node(node_id)
        node_sizes[node_id] = len(members)
    for src, targets in graph["links"].items():
        for tgt in targets:
            G.add_edge(src, tgt)

    if ax is not None and G.number_of_nodes() > 0:
        pos = nx.spring_layout(G, seed=42, k=1.5 / max(np.sqrt(G.number_of_nodes()), 1))
        sizes = np.array([node_sizes.get(n, 1) for n in G.nodes()])
        sizes = 30 + 300 * sizes / max(sizes.max(), 1)
        nx.draw_networkx(G, pos, ax=ax, node_size=sizes,
                         with_labels=False,
                         node_color="C0" if "Blob" in label else "C1",
                         edge_color="grey", alpha=0.8, width=0.8)
    if ax is not None:
        n_n, n_e = G.number_of_nodes(), G.number_of_edges()
        ax.set_title(f"{label}\n({n_n} nodes, {n_e} edges)")
        ax.axis("off")

    return G, graph


fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
G_blob, mg_blob = run_mapper(blob,     "Blob – Mapper",     ax=axes[0])
G_fil,  mg_fil  = run_mapper(filament, "Filament – Mapper", ax=axes[1])
plt.tight_layout(); plt.show()

for lbl, G in [("Blob", G_blob), ("Filament", G_fil)]:
    cc = list(nx.connected_components(G))
    largest = max(len(c) for c in cc) if cc else 0
    print(f"  {lbl}: nodes={G.number_of_nodes()}, edges={G.number_of_edges()}, "
          f"components={len(cc)}, largest component={largest}")

## 5. Density Diagnostic: kNN-Radius Sublevel H₀ (Secondary)

**Note:** this is a *density/cluster-tree* diagnostic, not a direct measure of geometric filamentness.
It detects multiple density modes (components born at different kNN-radius thresholds),
which can correlate with filamentary structure but is not the cleanest discriminator for shape.

Included for completeness; prefer the Rips/MST metrics above for blob-vs-filament classification.

In [ ]:
def sublevel_H0(X, k=K_NN):
    """
    H0 persistence via kNN-radius sublevel filtration.
    Union-Find on the kNN graph: vertices enter in order of increasing
    kNN-radius, edges appear when both endpoints are present.
    """
    rho = knn_radius(X, k)
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, idx = nn.kneighbors(X)
    order = np.argsort(rho)

    parent = np.arange(len(X))
    birth  = rho.copy()

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    bars = []
    present = np.zeros(len(X), dtype=bool)

    for v in order:
        present[v] = True
        for u in idx[v, 1:]:
            if not present[u]:
                continue
            rv, ru = find(v), find(u)
            if rv == ru:
                continue
            if birth[rv] < birth[ru]:
                rv, ru = ru, rv
            bars.append((birth[rv], rho[v]))
            parent[rv] = ru

    bars.append((rho[order[0]], np.inf))
    return np.array(bars)


def persistence_lengths(bars):
    finite = bars[np.isfinite(bars[:, 1])]
    return finite[:, 1] - finite[:, 0]


bars_blob = sublevel_H0(blob)
bars_fil  = sublevel_H0(filament)
pers_blob_sl = persistence_lengths(bars_blob)
pers_fil_sl  = persistence_lengths(bars_fil)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

N_SHOW = 50
for ax, pers, title, col in [
    (axes[0], pers_blob_sl, "Blob – top-50 sublevel H₀", "C0"),
    (axes[1], pers_fil_sl,  "Filament – top-50 sublevel H₀", "C1"),
]:
    sorted_p = np.sort(pers)[::-1][:N_SHOW]
    ax.barh(range(len(sorted_p)), sorted_p, color=col, alpha=0.7, height=0.8)
    ax.set_xlabel("Persistence (death − birth)")
    ax.set_ylabel("Bar rank"); ax.invert_yaxis()
    ax.set_title(title)

eps = 1e-8
combined = np.concatenate([pers_blob_sl[pers_blob_sl > eps],
                           pers_fil_sl[pers_fil_sl > eps]])
lo, hi = combined.min() * 0.5, combined.max() * 2
bins = np.geomspace(lo, hi, 40)
axes[2].hist(pers_blob_sl[pers_blob_sl > eps], bins=bins, alpha=0.6, label="Blob")
axes[2].hist(pers_fil_sl[pers_fil_sl > eps],   bins=bins, alpha=0.6, label="Filament")
axes[2].set_xscale("log"); axes[2].set_yscale("log")
axes[2].set_xlabel("Persistence (log)"); axes[2].set_ylabel("Count (log)")
axes[2].set_title("Sublevel H₀ distribution (density diagnostic)")
axes[2].legend()
plt.tight_layout(); plt.show()

print(f"Blob   – sublevel H₀: {len(pers_blob_sl):4d} bars,  "
      f"mean pers: {pers_blob_sl.mean():.6f},  max pers: {pers_blob_sl.max():.6f}")
print(f"Filament – sublevel H₀: {len(pers_fil_sl):4d} bars,  "
      f"mean pers: {pers_fil_sl.mean():.6f},  max pers: {pers_fil_sl.max():.6f}")

## 6. Summary Dashboard

Side-by-side comparison of all computed features.

In [ ]:
rows = []
for lbl, X, rips_res, mst_w, sublev_pers, G_map in [
    ("Blob",     blob,     rips_blob, mst_blob, pers_blob_sl, G_blob),
    ("Filament", filament, rips_fil,  mst_fil,  pers_fil_sl,  G_fil),
]:
    r0_n, r0_mu, r0_max, r0_p90 = rips_summary(rips_res, 0)
    r1_n, r1_mu, r1_max, r1_p90 = rips_summary(rips_res, 1)
    cc = list(nx.connected_components(G_map))
    d_est   = intrinsic_dim_knn(X)
    tail_r  = knn_radius_tail_ratio(X)

    rows.append(dict(
        dataset=lbl,
        rips_H0_bars=r0_n, rips_H0_mean=r0_mu, rips_H0_max=r0_max,
        rips_H1_bars=r1_n, rips_H1_mean=r1_mu, rips_H1_max=r1_max,
        mst_median=float(np.median(mst_w)),
        mst_p90_p50=float(np.percentile(mst_w, 90) / np.median(mst_w)),
        intrinsic_dim=d_est, knn_tail_ratio=tail_r,
        mapper_nodes=G_map.number_of_nodes(),
        mapper_edges=G_map.number_of_edges(),
        mapper_components=len(cc),
        sublev_H0_bars=len(sublev_pers),
        sublev_H0_mean=sublev_pers.mean(),
        sublev_H0_max=sublev_pers.max(),
    ))

header = f"{'Metric':36s} {'Blob':>12s} {'Filament':>12s}"
print(header)
print("=" * len(header))

key_labels = [
    (None,                "— PRIMARY: GEOMETRY —"),
    ("rips_H0_bars",      "  Rips H₀ #bars"),
    ("rips_H0_mean",      "  Rips H₀ mean pers"),
    ("rips_H0_max",       "  Rips H₀ max pers"),
    ("rips_H1_bars",      "  Rips H₁ #bars"),
    ("rips_H1_mean",      "  Rips H₁ mean pers"),
    ("rips_H1_max",       "  Rips H₁ max pers"),
    ("mst_median",        "  MST median edge weight"),
    ("mst_p90_p50",       "  MST p90/p50 ratio"),
    ("intrinsic_dim",     "  Intrinsic dimension est."),
    ("knn_tail_ratio",    "  kNN-radius p99/p50"),
    ("mapper_nodes",      "  Mapper nodes"),
    ("mapper_edges",      "  Mapper edges"),
    ("mapper_components", "  Mapper components"),
    (None,                "— SECONDARY: DENSITY —"),
    ("sublev_H0_bars",    "  Sublevel H₀ #bars"),
    ("sublev_H0_mean",    "  Sublevel H₀ mean pers"),
    ("sublev_H0_max",     "  Sublevel H₀ max pers"),
]
for key, nice in key_labels:
    if key is None:
        print(f"\n{nice}")
        continue
    v_b = rows[0][key]
    v_f = rows[1][key]
    fmt = ".4f" if isinstance(v_b, float) else "d"
    print(f"{nice:36s} {v_b:12{fmt}} {v_f:12{fmt}}")

---

## 7. Real Data: Tyson x0 Post-Burn-In Point Cloud

Load the post-burn-in x0 trajectory from the Tyson NNSE simulation and run the same TDA pipeline.
The point cloud lives in parameter space ($\mathbb{R}^{n_\text{params}}$); each row is the best-fit
parameter vector (position 0) at one MCMC-like step after burn-in.

**Question:** does this cloud look like a compact blob (well-constrained parameters) or a
filamentary/spiky manifold (sloppy directions)?

In [ ]:
# ============================================================================
# === LOAD TYSON x0 POST-BURN-IN DATA ===
# ============================================================================

import os

tyson_file = os.path.join(os.path.dirname(os.path.abspath("")), "tyson_x0_postburnin.npz")
if not os.path.exists(tyson_file):
    tyson_file = "tyson_x0_postburnin.npz"

data = np.load(tyson_file, allow_pickle=True)
tyson_X  = data["X"]           # (n_samples, n_params)
tyson_fX = data["fX"]          # (n_samples,)
tyson_steps = data["steps"]
tyson_param_names = list(data["param_names"])
tyson_p0 = data["p0_vec"]
tyson_burn_in = int(data["effective_burn_in"])

print(f"Loaded: {tyson_file}")
print(f"  Shape: {tyson_X.shape}  ({tyson_X.shape[0]} samples, {tyson_X.shape[1]} parameters)")
print(f"  Parameters: {tyson_param_names}")
print(f"  Burn-in step: {tyson_burn_in}")
print(f"  fX range: [{tyson_fX.min():.4f}, {tyson_fX.max():.4f}]")

# Normalize to [0,1]^D (each parameter divided by 2*p0, which is the sampling range)
tyson_norm = tyson_X / (2.0 * tyson_p0[None, :])
print(f"  Normalized range: [{tyson_norm.min():.3f}, {tyson_norm.max():.3f}]")

# 2-D PCA preview
tyson_2d = PCA(n_components=2).fit_transform(tyson_norm)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sc = axes[0].scatter(*tyson_2d.T, c=tyson_fX, s=8, alpha=0.6, cmap="viridis")
axes[0].set_title("Tyson x0 post-burn-in (PCA 2-D, coloured by fX)")
axes[0].set_xlabel("PC1"); axes[0].set_ylabel("PC2")
axes[0].set_aspect("equal")
plt.colorbar(sc, ax=axes[0], label="fX (squared diff)")

axes[1].scatter(*tyson_2d.T, c=tyson_steps, s=8, alpha=0.6, cmap="plasma")
axes[1].set_title("Coloured by step index")
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_aspect("equal")
plt.tight_layout(); plt.show()

### 7a. Vietoris–Rips PH + MST (Tyson x0)

In [ ]:
K_TYSON = min(K_NN, len(tyson_norm) // 5)

tyson_pca = PCA(n_components=min(PCA_DIM, tyson_norm.shape[1])).fit_transform(tyson_norm)

SUB_T = min(400, len(tyson_pca))
idx_t = np.random.choice(len(tyson_pca), SUB_T, replace=False)

rips_tyson = ripser(tyson_pca[idx_t], maxdim=1)
mst_tyson  = mst_edges(tyson_pca[idx_t])

fig = plt.figure(figsize=(16, 5))
gs_t = GridSpec(1, 3, figure=fig, wspace=0.3)

ax1 = fig.add_subplot(gs_t[0, 0])
plot_diagrams(rips_tyson["dgms"], ax=ax1, show=False)
ax1.set_title("Tyson x0 – Rips persistence diagram")

ax2 = fig.add_subplot(gs_t[0, 1])
bins_t = np.linspace(0, np.percentile(mst_tyson, 99), 50)
ax2.hist(mst_tyson, bins=bins_t, alpha=0.7, color="C2")
ax2.set_xlabel("MST edge weight"); ax2.set_ylabel("Count")
ax2.set_title("Tyson x0 – MST merge-distance")

ax3 = fig.add_subplot(gs_t[0, 2])
dgm0 = rips_tyson["dgms"][0]
fin0 = dgm0[np.isfinite(dgm0[:, 1])]
pers0 = np.sort(fin0[:, 1] - fin0[:, 0])[::-1][:50]
ax3.barh(np.arange(len(pers0)), pers0, height=0.8, color="C2", alpha=0.7)
ax3.invert_yaxis()
ax3.set_xlabel("Persistence"); ax3.set_ylabel("Bar rank")
ax3.set_title("Tyson x0 – Rips H₀ top-50 bars")
plt.show()

for dim in [0, 1]:
    n_bars, mu, mx, p90 = rips_summary(rips_tyson, dim)
    print(f"  Tyson x0 H{dim}: {n_bars:4d} finite bars, "
          f"mean pers = {mu:.4f}, max pers = {mx:.4f}")
print(f"  MST: median = {np.median(mst_tyson):.4f}, "
      f"p90/p50 = {np.percentile(mst_tyson, 90)/np.median(mst_tyson):.3f}")

### 7b. Intrinsic Dimension (Tyson x0)

In [ ]:
d_est_tyson = intrinsic_dim_knn(tyson_norm, k1=min(5, K_TYSON), k2=K_TYSON)
tail_tyson  = knn_radius_tail_ratio(tyson_norm, k=K_TYSON)

print(f"{'':20s} {'Intrinsic dim':>14s} {'kNN-radius p99/p50':>20s}")
print("-" * 56)
for lbl, d, r in [("Blob", intrinsic_dim_knn(blob), knn_radius_tail_ratio(blob)),
                   ("Filament", intrinsic_dim_knn(filament), knn_radius_tail_ratio(filament)),
                   ("Tyson x0", d_est_tyson, tail_tyson)]:
    print(f"  {lbl:16s} {d:14.2f} {r:20.3f}")

### 7c. Mapper Graph (Tyson x0)

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(7, 6))
G_tyson, mg_tyson = run_mapper(tyson_norm, "Tyson x0 – Mapper", k=K_TYSON, ax=ax)
plt.tight_layout(); plt.show()

cc_t = list(nx.connected_components(G_tyson))
largest_t = max(len(c) for c in cc_t) if cc_t else 0
print(f"  Tyson x0: nodes={G_tyson.number_of_nodes()}, edges={G_tyson.number_of_edges()}, "
      f"components={len(cc_t)}, largest component={largest_t}")

### 7d. Sublevel H₀ Density Diagnostic (Tyson x0, secondary)

In [ ]:
bars_tyson = sublevel_H0(tyson_norm, k=K_TYSON)
pers_tyson_sl = persistence_lengths(bars_tyson)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sorted_p = np.sort(pers_tyson_sl)[::-1][:50]
axes[0].barh(range(len(sorted_p)), sorted_p, color="C2", alpha=0.7, height=0.8)
axes[0].set_xlabel("Persistence"); axes[0].set_ylabel("Bar rank"); axes[0].invert_yaxis()
axes[0].set_title("Tyson x0 – sublevel H₀ top-50 (density)")

eps = 1e-8
pos = pers_tyson_sl[pers_tyson_sl > eps]
if len(pos):
    bins = np.geomspace(pos.min() * 0.5, pos.max() * 2, 40)
    axes[1].hist(pos, bins=bins, alpha=0.7, color="C2")
    axes[1].set_xscale("log"); axes[1].set_yscale("log")
axes[1].set_xlabel("Persistence (log)"); axes[1].set_ylabel("Count (log)")
axes[1].set_title("Tyson x0 – sublevel H₀ distribution")
plt.tight_layout(); plt.show()

print(f"Tyson x0 – sublevel H₀: {len(pers_tyson_sl)} bars, "
      f"mean = {pers_tyson_sl.mean():.6f}, max = {pers_tyson_sl.max():.6f}")

In [ ]:
r0_t = rips_summary(rips_tyson, 0)
r1_t = rips_summary(rips_tyson, 1)
cc_t = list(nx.connected_components(G_tyson))

tyson_row = dict(
    dataset="Tyson x0",
    rips_H0_bars=r0_t[0], rips_H0_mean=r0_t[1], rips_H0_max=r0_t[2],
    rips_H1_bars=r1_t[0], rips_H1_mean=r1_t[1], rips_H1_max=r1_t[2],
    mst_median=float(np.median(mst_tyson)),
    mst_p90_p50=float(np.percentile(mst_tyson, 90) / np.median(mst_tyson)),
    intrinsic_dim=d_est_tyson,
    knn_tail_ratio=tail_tyson,
    mapper_nodes=G_tyson.number_of_nodes(),
    mapper_edges=G_tyson.number_of_edges(),
    mapper_components=len(cc_t),
    sublev_H0_bars=len(pers_tyson_sl),
    sublev_H0_mean=pers_tyson_sl.mean(),
    sublev_H0_max=pers_tyson_sl.max(),
)

all_rows = rows + [tyson_row]

header = f"{'Metric':36s} {'Blob':>12s} {'Filament':>12s} {'Tyson x0':>12s}"
print(header)
print("=" * len(header))
for key, nice in key_labels:
    if key is None:
        print(f"\n{nice}")
        continue
    vals = [r[key] for r in all_rows]
    fmt = ".4f" if isinstance(vals[0], float) else "d"
    line = f"{nice:36s}"
    for v in vals:
        line += f" {v:12{fmt}}"
    print(line)